In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import numpy as np

from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoModel


In [3]:
class DataProcessor:

    def load_and_preprocess(self):
        """Load and preprocess the essays-big5 dataset from Hugging Face"""
        ds = load_dataset("jingjietan/essays-big5", split="train")
        texts = ds["text"]
        # Convert texts to a list of strings
        texts_list = [str(t) for t in texts]
        # Big-5 columns: 'O', 'C', 'E', 'A', 'N'
        big5_scores = np.array([
            [float(row["E"]), float(row["N"]), float(row["A"]), float(row["C"]), float(row["O"])]
            for row in ds
        ])
        # Normalize scores to [0, 1] (original scale is 1-5)
        big5_scores = (big5_scores - 1) / 4
        X_train, X_test, y_train, y_test = train_test_split(
            texts_list, big5_scores, test_size=0.2, random_state=42
        )
        return X_train, X_test, y_train, y_test

In [4]:
def mean_pool(last_hidden_state, attention_mask):
    # last_hidden_state: (B, T, H), attention_mask: (B, T)
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # (B, T, 1)
    summed = (last_hidden_state * mask).sum(dim=1)                  # (B, H)
    denom = mask.sum(dim=1).clamp(min=1e-6)                         # (B, 1)
    return summed / denom

class PersonalityClassifier(nn.Module):
    """
    Faster version:
      - No SentenceTransformer.encode() in forward.
      - SBERT backbone is an AutoModel run on GPU with batched tokenized inputs.
      - You must provide *two* tokenized inputs in each batch:
          (roberta_input_ids, roberta_attention_mask)
          (sbert_input_ids,   sbert_attention_mask)
    """
    def __init__(
        self,
        roberta_name: str = "roberta-base",
        sbert_name: str = "sentence-transformers/all-mpnet-base-v2",
        proj_dim: int = 384,          # set to None to skip projection
        dropout: float = 0.2
    ):
        super().__init__()
        # Backbones
        self.roberta = AutoModel.from_pretrained(roberta_name)
        self.sbert   = AutoModel.from_pretrained(sbert_name)

        # Freeze (you can unfreeze later if needed)
        for p in self.roberta.parameters():
            p.requires_grad = False
        for p in self.sbert.parameters():
            p.requires_grad = False

        roberta_dim = self.roberta.config.hidden_size  # typically 768
        sbert_dim   = self.sbert.config.hidden_size    # mpnet-base = 768

        # Optional lightweight projections to shrink classifier size
        if proj_dim is not None:
            self.roberta_proj = nn.Sequential(
                nn.Linear(roberta_dim, proj_dim), nn.ReLU(), nn.LayerNorm(proj_dim)
            )
            self.sbert_proj = nn.Sequential(
                nn.Linear(sbert_dim, proj_dim), nn.ReLU(), nn.LayerNorm(proj_dim)
            )
            combined_dim = proj_dim * 2
        else:
            self.roberta_proj = nn.Identity()
            self.sbert_proj   = nn.Identity()
            combined_dim = roberta_dim + sbert_dim

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 5)  # VAD
        )

    def forward(
        self,
        roberta_input_ids,
        roberta_attention_mask,
        sbert_input_ids,
        sbert_attention_mask,
    ):
        # RoBERTa embeddings (mean pooled)
        r_out = self.roberta(input_ids=roberta_input_ids,
                             attention_mask=roberta_attention_mask)
        r_embed = mean_pool(r_out.last_hidden_state, roberta_attention_mask)
        r_embed = self.roberta_proj(r_embed)

        # SBERT/MPNet embeddings (mean pooled)
        s_out = self.sbert(input_ids=sbert_input_ids,
                           attention_mask=sbert_attention_mask)
        s_embed = mean_pool(s_out.last_hidden_state, sbert_attention_mask)
        s_embed = self.sbert_proj(s_embed)

        # Concatenate
        combined = torch.cat([r_embed, s_embed], dim=1)

        # Predict VAD
        return self.classifier(combined)


In [ ]:
class TextPersonalityDataset(Dataset):
    def __init__(self, texts, labels, roberta_tokenizer, sbert_tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.roberta_tokenizer = roberta_tokenizer
        self.sbert_tokenizer = sbert_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        # RoBERTa
        roberta_enc = self.roberta_tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        # sBERT
        sbert_enc = self.sbert_tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'text': text,
            'roberta_input_ids': roberta_enc['input_ids'].squeeze(),
            'roberta_attention_mask': roberta_enc['attention_mask'].squeeze(),
            'sbert_input_ids': sbert_enc['input_ids'].squeeze(),
            'sbert_attention_mask': sbert_enc['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.float32)
        }

def train_model(model, train_loader, val_loader, device, num_epochs=10):
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                     factor=0.5, patience=1)

    best_val_loss = float('inf')
    patience = 2
    wait = 0
    best_epoch = 0

    trait_names = ['E', 'N', 'A', 'C', 'O']

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            roberta_input_ids = batch['roberta_input_ids'].to(device)
            roberta_attention_mask = batch['roberta_attention_mask'].to(device)
            sbert_input_ids = batch['sbert_input_ids'].to(device)
            sbert_attention_mask = batch['sbert_attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(
                roberta_input_ids,
                roberta_attention_mask,
                sbert_input_ids,
                sbert_attention_mask
            )
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # ---- Validation loss ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                roberta_input_ids = batch['roberta_input_ids'].to(device)
                roberta_attention_mask = batch['roberta_attention_mask'].to(device)
                sbert_input_ids = batch['sbert_input_ids'].to(device)
                sbert_attention_mask = batch['sbert_attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    roberta_input_ids,
                    roberta_attention_mask,
                    sbert_input_ids,
                    sbert_attention_mask
                )
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        def trait_accuracy(preds, targets):
            return 100.0 - np.mean(np.abs(preds - targets)) * 100.0



        def trait_error(preds, targets):
            # overall mean absolute error across all traits
            return np.mean(np.abs(preds - targets))

        # train preds/targets
        train_preds = []
        train_targets = []
        model.eval()
        with torch.no_grad():
            for batch in train_loader:
                roberta_input_ids = batch['roberta_input_ids'].to(device)
                roberta_attention_mask = batch['roberta_attention_mask'].to(device)
                sbert_input_ids = batch['sbert_input_ids'].to(device)
                sbert_attention_mask = batch['sbert_attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    roberta_input_ids,
                    roberta_attention_mask,
                    sbert_input_ids,
                    sbert_attention_mask
                )
                train_preds.append(outputs.cpu().numpy())
                train_targets.append(labels.cpu().numpy())

        train_preds = np.concatenate(train_preds, axis=0)
        train_targets = np.concatenate(train_targets, axis=0)
        train_acc = trait_accuracy(train_preds, train_targets)
        train_err = trait_error(train_preds, train_targets)

        val_preds = []
        val_targets = []
        with torch.no_grad():
            for batch in val_loader:
                roberta_input_ids = batch['roberta_input_ids'].to(device)
                roberta_attention_mask = batch['roberta_attention_mask'].to(device)
                sbert_input_ids = batch['sbert_input_ids'].to(device)
                sbert_attention_mask = batch['sbert_attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    roberta_input_ids,
                    roberta_attention_mask,
                    sbert_input_ids,
                    sbert_attention_mask
                )
                val_preds.append(outputs.cpu().numpy())
                val_targets.append(labels.cpu().numpy())

        val_preds = np.concatenate(val_preds, axis=0)
        val_targets = np.concatenate(val_targets, axis=0)
        val_err = trait_error(val_preds, val_targets)



        per_trait_train_err = np.mean(np.abs(train_preds - train_targets), axis=0)
        per_trait_val_err   = np.mean(np.abs(val_preds   - val_targets),   axis=0)

        per_trait_train_acc = 100.0 - per_trait_train_err * 100.0
        per_trait_val_acc   = 100.0 - per_trait_val_err   * 100.0

        # Which trait is best / worst on validation?
        best_idx  = np.argmin(per_trait_val_err)
        worst_idx = np.argmax(per_trait_val_err)

        # ---- Logging ----
        print(
            f"Epoch {epoch+1}: "
            f"Train Acc: {train_acc:.4f}% | Train Loss: {avg_train_loss:.4f}, Train MAE: {train_err:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}, Val MAE: {val_err:.4f}"
        )

        train_trait_str = " ".join(
            f"{name}:MAE={err:.4f},Acc={acc:.2f}%"
            for name, err, acc in zip(trait_names, per_trait_train_err, per_trait_train_acc)
        )
        val_trait_str = " ".join(
            f"{name}:MAE={err:.4f},Acc={acc:.2f}%"
            for name, err, acc in zip(trait_names, per_trait_val_err, per_trait_val_acc)
        )

        print(f"  Train per-trait  -> {train_trait_str}")
        print(f"  Val per-trait    -> {val_trait_str}")
        print(
            f"  Best trait (val):  {trait_names[best_idx]} "
            f"(MAE={per_trait_val_err[best_idx]:.4f}) | "
            f"Worst trait (val): {trait_names[worst_idx]} "
            f"(MAE={per_trait_val_err[worst_idx]:.4f})"
        )

        # scheduler / early stopping
        scheduler.step(avg_val_loss)

        # Save weights from epoch 3 or 4 (as before)
        if epoch + 1 in [3, 4]:
            torch.save(model.state_dict(), f'weights_epoch_{epoch+1}.pt')

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model.pt')
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping at epoch {epoch+1}, best epoch = {best_epoch+1}')
                break


def main():
    data_processor = DataProcessor()
    X_train, X_test, y_train, y_test = data_processor.load_and_preprocess()

    roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
    sbert_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
    model = PersonalityClassifier()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    train_dataset = TextPersonalityDataset(X_train, y_train, roberta_tokenizer, sbert_tokenizer)
    test_dataset = TextPersonalityDataset(X_test, y_test, roberta_tokenizer, sbert_tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16)

    train_model(model, train_loader, test_loader, device)

if __name__ == '__main__':
    main()


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: Train Acc: 87.9085% | Train Loss: 0.0226, Train MAE: 0.1209 | Val Loss: 0.0169, Val MAE: 0.1240
  Train per-trait  -> E:MAE=0.1221,Acc=87.79% N:MAE=0.1208,Acc=87.92% A:MAE=0.1218,Acc=87.82% C:MAE=0.1221,Acc=87.79% O:MAE=0.1177,Acc=88.23%
  Val per-trait    -> E:MAE=0.1250,Acc=87.50% N:MAE=0.1253,Acc=87.47% A:MAE=0.1218,Acc=87.82% C:MAE=0.1239,Acc=87.61% O:MAE=0.1240,Acc=87.60%
  Best trait (val):  A (MAE=0.1218) | Worst trait (val): N (MAE=0.1253)
